# 01. 量化基础知识

> 说明：在官方文档中，**AMD Quark** 有时会简称为 **Quark**。除非另有说明，文中的 “Quark” 特指 AMD Quark 量化器，不要和其他同名产品或技术混淆。

> 本章配图缓存到本课程的 `assets/intro/` 目录，便于离线打开 notebook。

## 什么是量化？

量化指的是：用较低 bit 宽度的数值表示近似替代较高 bit 宽度的数值表示，从而降低存储、带宽和计算成本。

一个简单的 bit 宽度例子是整数 `1024`。它可以被 `int32` 表示，也可以被 `int16` 无损表示，因为 `1024` 没有超出 `int16` 的取值范围。这个例子说明了降低 bit 宽度可以减少存储开销，但神经网络量化通常更复杂：权重和激活大多是浮点数，低 bit 表示往往只能近似原始值。

在模型量化中，常见做法是用 scale、zero point、observer 和校准数据，把浮点 tensor 映射到低 bit 整数或低 bit 浮点格式，并控制由此引入的误差。


### 整数 bit 宽度压缩

下面两张图展示 `1024` 的 bit 宽度示例：同一个整数在 `32 bit` 表示中包含大量高位 `0`，转成 `16 bit` 后仍可无损表达。

![1024 的 32-bit 表示](assets/intro/bit_1024_32.png)

![从 32-bit 压缩到 16-bit](assets/intro/bit_32_to_16.png)


### 浮点数为什么更麻烦

浮点数通常包含符号位、指数位和尾数位。它不像整数那样容易直接“砍掉高位 0”，所以模型量化需要 scale、zero point、observer、校准数据等机制来控制误差。

![浮点数表示示意](assets/intro/floating_point.png)

做量化时，需要关注几个问题：

- **量化通常是有损压缩**：例如 `1024.23` 在较低 bit 宽度下可能只能近似为 `1024.0` 或 `1024.5`。
- **低 bit 运算可能溢出**：例如 `255` 可用 `8 bit` 表示，但 `255 * 2 = 510` 需要至少 `9 bit`。
- **数据类型和硬件支持相关**：量化格式可能是 INT8/INT4、FP8、MXFP4 等；是否能带来性能收益，取决于目标硬件和推理框架是否有对应 kernel。

AMD Quark 的作用是把量化配置、校准、observer、导出等流程封装起来，减少手写底层量化逻辑的成本。


## 模型量化

对于深度学习模型，量化通常指把模型的 **权重** 和/或 **激活值** 从高精度浮点表示转换为较低 bit 宽度的浮点或整数表示。

量化可以降低模型的内存占用、显存带宽和计算成本；代价是可能引入精度损失。实际部署时，需要在 **精度、吞吐、延迟、模型大小** 之间做取舍。

模型量化技术大致分为两类：

- **量化感知训练，Quantization-Aware Training，QAT**：在训练过程中引入量化模拟，让模型适应量化误差。
- **训练后量化，Post-Training Quantization，PTQ**：在模型训练完成后应用量化，不重新训练或只做少量校准。

LLM 部署中常见的是 PTQ，因为它能直接作用于已经训练完成的模型。Quark 提供 SmoothQuant、AWQ、GPTQ、Rotation 等训练后量化算法，用于降低量化带来的精度损失。


从模型的量化表示来看，也可以大致分为两类：

- **量化到低 bit 浮点精度**：例如从 `float32` 转换到 `float16`、`bfloat16` 或 `float8`。这种情况下，数据仍然保持类似的浮点格式，只是 bit 宽度更低。当硬件支持这些低 bit 浮点数据类型时，这种方式会很有优势。
- **量化到整数精度**：例如从 `float32` 或 `float16` 转换到 `int8` 或 `int4`。这种情况下，数据会被映射到整数范围内。当硬件支持整数数据类型，并且能比浮点类型更高效地执行整数运算时，这种方式会更有收益。

## 量化时需要考虑什么？

选择量化方案时，需要关注以下因素：

- **目标硬件和推理框架**：不同 CPU/GPU/NPU 对 INT、FP8、MXFP4 等低精度格式的支持不同。
- **模型结构和数值分布**：activation outlier、layer 类型、KV cache、MoE 结构等都会影响量化难度。
- **精度要求**：不同任务对困惑度、准确率、生成质量和稳定性的敏感程度不同。
- **导出格式**：同一个量化配置可以导出为不同格式；后续消费工具必须能正确读取该格式。


## 整数量化

将真实数值映射到整数，需要计算量化参数，通常称为 **scale** 和 **offset**。

量化公式：

```text
q = round(r / s + z)
```

反量化公式：

```text
r = (q - z) * s
```

在这些公式中：

- `q` 是量化后的值。
- `r` 是真实值。
- `s` 是 scale。
- `z` 是 offset。

为了计算这些量化参数，需要观察 tensor 的真实取值，并确定它的最小值和最大值。随后，scale 和 offset 会根据这些值计算出来。

Quark 提供了多种算法，帮助你调整这些量化参数，从而达到想要的精度和性能平衡。

量化可以根据 offset / zero point 的取值进行分类：

- **对称量化，symmetric quantization**：`z = 0`
- **非对称量化，asymmetric quantization**：`z != 0`

根据 scale 和 offset 的计算粒度，量化还可以分为：

- **per-tensor**：整个 tensor 共用一组量化参数。
- **per-channel**：每个 channel 使用一组量化参数。
- **per-group / per-block**：每个 group 或 block 使用一组量化参数，例如 INT4 group-wise、MXFP4 block scaling。

Quark for PyTorch 和 Quark for ONNX 支持的量化 scheme 不完全相同，实际使用时应以对应后端的配置文档和脚本参数为准。


## 伪量化，Fake Quantization

**Fake quantization** 用于在高精度计算图中模拟低精度量化效果。它通常先把数值映射到目标低 bit 格式，再反量化回当前计算 dtype，从而在不依赖目标低精度硬件 kernel 的情况下评估量化误差。

在 PTQ 流程中，fake quantization 常用于校准、误差评估和量化算法搜索。最终导出时，模型是否真的以低 bit 权重存储，取决于选择的导出格式和 `export_weight_format` 等参数。


### 图示：Fake Quantization

Fake quantization 的核心是“模拟低精度”：先按目标低 bit 格式量化，再反量化回原始计算 dtype。这样可以在当前硬件上近似评估低精度模型的精度影响。

![Fake Quantization 示意](assets/intro/fake_quantize.png)

## 这意味着什么？

Fake quantization 阶段并不等同于最终部署格式。例如，一个 `float16` 权重在校准时可能仍以高精度 tensor 参与计算，只是在前向过程中插入量化/反量化逻辑来模拟低 bit 误差。

这样做的好处是：即使当前硬件还没有目标低 bit kernel，也可以先评估量化对精度的影响。真正用于部署的模型大小和计算方式，则由后续导出格式以及推理框架决定。


## 数值什么时候真正转换为量化数据类型？

在 Quark 中，量化流程通常可以分成两个阶段：

1. **校准和量化模拟**：通过 observer、fake quantizer 和校准数据估计量化参数，并模拟低 bit 误差。
2. **导出和部署**：根据目标格式导出模型，例如 QDQ 图、Hugging Face safetensors、ONNX 或 GGUF 等。

如果导出为 QDQ 模型，计算图会显式插入 Quantize / DeQuantize 节点，用节点表达量化行为。如果导出为 Quark Hugging Face 格式，则会保存量化权重、量化配置和必要的元数据，后续由支持 Quark quantization 的推理框架读取。

因此，“是否真正存成低 bit”不是由 fake quantization 本身决定，而是由导出格式、权重格式和推理后端共同决定。


### 原始节点与 QDQ 节点

QDQ 模型会在原始计算图周围插入 Quantize / DeQuantize 节点，用显式节点表达量化行为。

![原始节点](assets/intro/nodes_original.png)

![插入 QDQ 后的节点](assets/intro/nodes_qdq.png)

## Quark 内部量化时发生了什么？

当模型进入 Quark 量化流程时，Quark 会根据配置找到可量化层，并替换为带量化逻辑的等价层。

常见可替换层包括：

- `Linear`
- `Conv2d`

例如，`Linear` 可以被替换为 `QuantLinear`。这些 Quant 层会根据配置在输入、输出、权重或 bias 上挂接 observer / fake quantizer。

随后，校准数据会通过模型执行前向传播。observer 记录 tensor 的统计信息，例如最小值、最大值、分位数或 block 级 scale，用于计算量化参数。

常见 observer 包括：

- `PerTensorMinMaxObserver`
- `PerChannelMinMaxObserver`
- `PerBlockMXObserver`


### 图示：Quark 替换层并插入量化逻辑

Quark 会把可量化层替换成对应的 Quant 版本，例如 `Linear -> QuantLinear`。这些 Quant 层可以在输入、输出、权重或 bias 上挂接 fake quantizer / observer，从而完成校准和量化模拟。

![Quark 层替换示意](assets/intro/layer_change.png)

## 本章小结

本章需要掌握的核心概念：

- 量化用更低 bit 宽度表示数值，以降低模型大小、显存带宽和计算成本。
- 模型量化通常是有损近似，需要在精度和性能之间取舍。
- LLM 部署中常见的是 PTQ，因为它可以作用于已经训练完成的模型。
- 整数量化依赖 `scale` 和 `zero point / offset`；MXFP4 等格式还会使用 block scale。
- Fake quantization 用于模拟低 bit 误差，不等同于最终部署时的真实存储格式。
- Quark 会根据配置替换部分层，并通过 observer 和 calibration data 统计量化参数。

下一章将把这些基础概念连接到 Quark for PyTorch 的 LLM 配置系统。
